# 01 Extract From Bigquery

Flood-It! retention and churn project. Run cells top to bottom.

### Imports

In [ ]:
import pandas as pd                                   # tables
from pathlib import Path                              # file paths that work on every operating system
from google.cloud import bigquery                     # official BigQuery client for Python
import pydata_google_auth                             # opens a browser so you can sign in with your Google account

### Settings and paths

In [ ]:
PROJECT_ID = "your-project-id"                        # REPLACE with your Google Cloud project ID (shown in the BigQuery console)
ROOT = Path.cwd()                                     # folder the notebook runs in
if ROOT.name == "notebooks":                          # if we are inside notebooks/...
    ROOT = ROOT.parent                                # ...go up to the project root
SQL_DIR = ROOT / "sql"                                # where the .sql files live
OUT_DIR = ROOT / "data" / "processed"                 # where exported tables will be saved
OUT_DIR.mkdir(parents=True, exist_ok=True)            # create the folder if it does not exist

### Sign in and connect (a browser window opens the first time; the login is cached afterwards)

In [ ]:
SCOPES = ["https://www.googleapis.com/auth/cloud-platform"]          # permission to use BigQuery on your behalf
credentials = pydata_google_auth.get_user_credentials(SCOPES)        # browser sign-in, then cached on your computer
client = bigquery.Client(project=PROJECT_ID, credentials=credentials)  # the connection every query will use
print(client.query("SELECT 'connected' AS status").to_dataframe())  # quick test: should print one row saying connected

### Run the exploration queries (01-05) and keep their results as evidence

In [ ]:
explore_files = sorted(SQL_DIR.glob("0[1-5]_*.sql"))                 # files 01 to 05
for sql_file in explore_files:                                       # loop over them one by one
    result = client.query(sql_file.read_text()).to_dataframe()      # run the query and download the result
    result.to_csv(OUT_DIR / f"explore_{sql_file.stem}.csv", index=False)  # save as CSV
    print(sql_file.name, result.shape)                               # confirm rows and columns
    print(result.head(10).to_string(index=False))                    # show the first rows

### Create the dataset (00) and all views (06-17)

In [ ]:
build_files = [SQL_DIR / "00_create_dataset.sql"] + sorted(SQL_DIR.glob("0[6-9]_*.sql")) + sorted(SQL_DIR.glob("1[0-7]_*.sql"))  # build order matters
for sql_file in build_files:                                         # loop over the files in order
    job = client.query(sql_file.read_text())                         # send the CREATE statement to BigQuery
    job.result()                                                     # wait until BigQuery finishes
    print("created:", sql_file.name)                                 # confirm

### Export the analysis views to CSV for the Python notebooks

In [ ]:
views = ["kpi_daily", "retention_summary", "retention_cohorts_weekly", "retention_by_milestone",
         "first_day_funnel", "retention_by_segment", "player_features"]          # tables the later notebooks read
for view in views:                                                               # loop over each view
    table = client.query(f"SELECT * FROM `{PROJECT_ID}.flood_it.{view}`").to_dataframe()  # download the whole view
    table.to_csv(OUT_DIR / f"{view}.csv", index=False)                           # save it
    print(view, table.shape)                                                     # rows and columns